In [ ]:
# =========================
# 1. INSTALL LIBRARIES
# =========================
!pip install xgboost joblib -q

In [ ]:
# =========================
# 2. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import joblib

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [ ]:
# =========================
# 3. LOAD DATASET
# =========================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

df.head()

Saving training_data.csv to training_data.csv


,post_id,day_of_week,hour_of_day,is_weekend,content_length,has_image,has_video,reactions_count,comments_count,saves_count,engagement_score
0,69e8e5288aa53129cfe9f7aa,4,8,0,48,0,0,0,3,0,6
1,69e8e52a8aa53129cfe9f7b9,1,22,0,75,0,0,45,12,10,109
2,69e8e53a8aa53129cfe9f874,1,11,0,135,0,0,9,2,0,13
3,69e8e53d8aa53129cfe9f899,6,6,1,133,0,0,1,3,0,7
4,69e8e53f8aa53129cfe9f8a9,5,7,0,132,0,0,3,3,0,9


In [ ]:
# =========================
# 4. CHECK DATA
# =========================
print("Shape:", df.shape)
print(df.columns)
print(df.isnull().sum())

Shape: (201, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   post_id           201 non-null    object
 1   day_of_week       201 non-null    int64 
 2   hour_of_day       201 non-null    int64 
 3   is_weekend        201 non-null    int64 
 4   content_length    201 non-null    int64 
 5   has_image         201 non-null    int64 
 6   has_video         201 non-null    int64 
 7   reactions_count   201 non-null    int64 
 8   comments_count    201 non-null    int64 
 9   saves_count       201 non-null    int64 
 10  engagement_score  201 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 17.4+ KB
None
post_id             0
day_of_week         0
hour_of_day         0
is_weekend          0
content_length      0
has_image           0
has_video           0
reactions_count     0
comments_count      0
saves_count         0
eng

,day_of_week,hour_of_day,is_weekend,content_length,has_image,has_video,reactions_count,comments_count,saves_count,engagement_score
count,201.000000,201.000000,201.000000,201.000000,201.0,201.0,201.000000,201.000000,201.000000,201.000000
mean,2.810945,11.492537,0.233831,100.895522,0.0,0.0,7.726368,3.636816,1.756219,22.024876
std,1.893167,7.192440,0.424323,47.848449,0.0,0.0,12.428988,5.894272,2.815186,31.794880
min,0.000000,0.000000,0.000000,15.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,1.000000,5.000000,0.000000,61.000000,0.0,0.0,1.000000,0.000000,0.000000,1.000000
50%,3.000000,11.000000,0.000000,105.000000,0.0,0.0,3.000000,2.000000,1.000000,11.000000
75%,4.000000,18.000000,0.000000,136.000000,0.0,0.0,7.000000,4.000000,2.000000,19.000000
max,6.000000,23.000000,1.000000,214.000000,0.0,0.0,57.000000,32.000000,12.000000,142.000000


In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

In [ ]:
# =========================
# 5. DEFINE FEATURES AND TARGET
# =========================
FEATURES = [
    "is_weekend",
    "content_length",
    "has_image",
    "has_video",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

TARGET = "engagement_score"

X = df[FEATURES]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (201, 9)
y shape: (201,)


In [ ]:
# =========================
# 6. TRAIN / TEST SPLIT
# =========================

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train size: (160, 9)
Test size: (41, 9)


In [ ]:
# =========================
# 7. TRAIN XGBOOST MODEL
# =========================

model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# =========================
# 8. EVALUATE MODEL
# =========================

def evaluate_model(name, X_data, y_data):
    pred = model.predict(X_data)

    mae = mean_absolute_error(y_data, pred)
    rmse = np.sqrt(mean_squared_error(y_data, pred))
    r2 = r2_score(y_data, pred)

    print(f"===== {name} =====")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

evaluate_model("Validation", X_val, y_val)
evaluate_model("Test", X_test, y_test)

MAE: 2.5133094787597656
RMSE: 6.611086360055163
R2 Score: 0.9651683568954468


In [ ]:
# =========================
# 9. FEATURE IMPORTANCE
# =========================

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df

,feature,importance
8,saves_count,0.604104
6,reactions_count,0.216717
7,comments_count,0.164140
3,content_length,0.006264
0,day_of_week,0.004313
1,hour_of_day,0.004143
2,is_weekend,0.000319
4,has_image,0.000000
5,has_video,0.000000


In [ ]:
# =========================
# 10. Global baseline for new user
# =========================

global_hour_baseline = (
    df.groupby("hour_of_day")[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: "avg_engagement_score"})
    .sort_values(by="avg_engagement_score", ascending=False)
)

global_hour_baseline.head(10)

def get_global_best_time(top_n=3):
    result = global_hour_baseline.head(top_n)

    return [
        {
            "hour": int(row["hour_of_day"]),
            "score": float(row["avg_engagement_score"]),
            "source": "global_baseline"
        }
        for _, row in result.iterrows()
    ]

In [ ]:
# =========================
# 11. PREDICT BEST TIME TO POST
# =========================

def build_prediction_row(day_of_week, hour, content_length, has_image, has_video):
    is_weekend = 1 if day_of_week in [5, 6] else 0

    return {
        "is_weekend": is_weekend,
        "content_length": content_length,
        "has_image": has_image,
        "has_video": has_video,
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "day_sin": np.sin(2 * np.pi * day_of_week / 7),
        "day_cos": np.cos(2 * np.pi * day_of_week / 7),
    }


def predict_personalized_best_time(
    day_of_week,
    content_length,
    has_image,
    has_video,
    candidate_hours=None,
    top_n=3
):
    if candidate_hours is None:
        candidate_hours = [8, 9, 10, 11, 12, 14, 16, 18, 19, 20, 21, 22]

    rows = [
        build_prediction_row(day_of_week, hour, content_length, has_image, has_video)
        for hour in candidate_hours
    ]

    predict_df = pd.DataFrame(rows)
    scores = model.predict(predict_df)

    result = pd.DataFrame({
        "hour": candidate_hours,
        "predicted_score": scores
    }).sort_values(by="predicted_score", ascending=False)

    return [
        {
            "hour": int(row["hour"]),
            "score": float(row["predicted_score"]),
            "source": "personalized_ml"
        }
        for _, row in result.head(top_n).iterrows()
    ]

In [ ]:
def suggest_best_time_to_post(
    user_post_count,
    day_of_week,
    content_length,
    has_image,
    has_video
):
    if user_post_count == 0:
        top_slots = get_global_best_time()

        return {
            "mode": "global_baseline",
            "message": "User chưa có dữ liệu cá nhân, hệ thống dùng xu hướng chung của nền tảng.",
            "best_time": top_slots[0],
            "top_slots": top_slots
        }

    if user_post_count < 5:
        global_slots = get_global_best_time()
        ml_slots = predict_personalized_best_time(
            day_of_week=day_of_week,
            content_length=content_length,
            has_image=has_image,
            has_video=has_video
        )

        return {
            "mode": "hybrid_fallback",
            "message": "User có ít dữ liệu, hệ thống kết hợp global baseline và ML.",
            "best_time": global_slots[0],
            "global_slots": global_slots,
            "ml_slots": ml_slots
        }

    top_slots = predict_personalized_best_time(
        day_of_week=day_of_week,
        content_length=content_length,
        has_image=has_image,
        has_video=has_video
    )

    return {
        "mode": "personalized_ml",
        "message": "Dự đoán dựa trên mô hình ML.",
        "best_time": top_slots[0],
        "top_slots": top_slots
    }

,hour_of_day,predicted_score
11,22,0.059086
9,20,0.059086
10,21,0.059086


In [ ]:
suggest_best_time_to_post(
    user_post_count=0,
    day_of_week=5,
    content_length=100,
    has_image=1,
    has_video=0
)

In [ ]:
suggest_best_time_to_post(
    user_post_count=10,
    day_of_week=5,
    content_length=100,
    has_image=1,
    has_video=0
)

In [ ]:
joblib.dump(model, "best_time_to_post_xgboost_fixed.pkl")
joblib.dump(global_hour_baseline, "global_hour_baseline.pkl")
joblib.dump(FEATURES, "best_time_features.pkl")

files.download("best_time_to_post_xgboost_fixed.pkl")
files.download("global_hour_baseline.pkl")
files.download("best_time_features.pkl")

Model saved: best_time_to_post_xgboost.pkl
